# 🧪 Gestão da Qualidade Laboratorial: Setor de Análises Biológicas
### Microbiologia · Urinálise · Parasitologia · Micologia
**Autora:** Luanna Oliveira (Biomédica & Estudante de Sistemas de Informação)  
**Foco de Negócio:** Redução de Recoletas, Cumprimento de SLA/TAT e Eliminação de Erros de Laudo

---


## 💡 1. Entendendo a Tecnologia: Como o Python e o Pandas Funcionam?

Como estudante de **Sistemas de Informação**, é fundamental entender por que usamos **Pandas** e como os dados são manipulados na memória do computador:

1. **Dicionário (`dict`):** Uma coleção de pares `chave: valor`. No nosso caso, cada dicionário representa **uma ocorrência de não conformidade**.
2. **Lista (`list`):** Uma sequência ordenada de itens. Nossa lista agrupa múltiplos dicionários (linhas de desvio).
3. **DataFrame do Pandas:** É a conversão dessa lista em uma **estrutura bidimensional (tabela)** com linhas, colunas tipadas e índices.

> **Vantagem em TI:** Enquanto o Excel é manual e limitado a 1 milhão de linhas, o script em Pandas é **automatizável**, **auditável**, e integra perfeitamente com pipelines de dados (ETL) e bancos de dados SQL.

---


## 📋 2. O Setor Integrado: Amostras e Ocorrências Mapeadas

O laboratório lida com 4 grandes frentes, sendo a **Urina** a amostra de maior volume:

| Especialidade | Amostras Principais | Desvios Mais Frequentes |
| :--- | :--- | :--- |
| **Urinálise** | Urina recente (jato médio) | Tubo cônico rachado na centrifugação; frasco com vazamento; volume insuficiente (< 10 mL); urina parada > 2h sem refrigeração. |
| **Microbiologia** | Urocultura, Secreções, Hemoculturas | Cadastro trocado (cadastrar EAS no lugar de Urocultura); frasco não estéril; contaminação da amostra; repique por cultura mista. |
| **Parasitologia** | Fezes frescas e conservadas | Frasco sem conservante (MIF/SAF) para pesquisa de trofozoítos; conservante derramado/vazando; material misturado com urina. |
| **Micologia** | Raspado de unha, pele e pelos | Quantidade insuficiente de escamas dérmicas para exame a fresco (KOH) e cultura; amostra colhida com resíduo de esmalte/pomada. |
| **Recepção / Geral** | Todas as amostras | **Cadastro de Exame Errado** na recepção; tubo ou frasco sem etiqueta de código de barras. |

---


## 🐍 3. Construindo o Catálogo de Falhas em Python (Código Comentado Passo a Passo)
Execute a célula abaixo para ver o Pandas transformar nossa lista de regras em um DataFrame estruturado.

In [ ]:
# PASSO 1: Importar a biblioteca Pandas com o apelido padrão "pd"
import pandas as pd

# PASSO 2: Criar a lista de dicionários com os motivos de recoleta, atrasos e erros
ocorrencias_laboratorio = [
    # --- RECOLETAS / REJEIÇÕES (PRÉ-ANALÍTICA) ---
    {
        "pilar": "Recoleta / Rejeição",
        "setor": "Urinálise",
        "amostra": "Urina",
        "motivo": "Amostra Não Vedada / Vazamento",
        "descricao": "Frasco coletor frouxo com derramamento do líquido durante o transporte",
        "origem_falha": "Coleta / Recepção",
        "gravidade": "Alta",
        "sla_dias": 1
    },
    {
        "pilar": "Recoleta / Rejeição",
        "setor": "Urinálise",
        "amostra": "Urina",
        "motivo": "Tubo Rachado / Quebrado",
        "descricao": "Tubo cônico trincado durante transporte ou centrifugação com perda de amostra",
        "origem_falha": "Logística / Bancada",
        "gravidade": "Alta",
        "sla_dias": 1
    },
    {
        "pilar": "Recoleta / Rejeição",
        "setor": "Geral / Recepção",
        "amostra": "Diversas",
        "motivo": "Cadastro de Exame Errado",
        "descricao": "Atendente cadastrou exame divergente do pedido médico (ex: cadastrou EAS e omitiu Urocultura)",
        "origem_falha": "Recepção / Atendimento",
        "gravidade": "Crítica",
        "sla_dias": 1
    },
    {
        "pilar": "Recoleta / Rejeição",
        "setor": "Parasitologia",
        "amostra": "Fezes",
        "motivo": "Conservante Inadequado / Ausente",
        "descricao": "Fezes enviadas sem líquido conservante (MIF/SAF) para pesquisa de trofozoítos frágeis",
        "origem_falha": "Coleta / Paciente",
        "gravidade": "Média",
        "sla_dias": 2
    },
    {
        "pilar": "Recoleta / Rejeição",
        "setor": "Micologia",
        "amostra": "Raspado Dérmico / Unha",
        "motivo": "Volume Insuficiente",
        "descricao": "Quantidade de escamas insuficiente para realizar o exame direto (KOH) e o cultivo",
        "origem_falha": "Coleta / Posto de Coleta",
        "gravidade": "Média",
        "sla_dias": 2
    },
    {
        "pilar": "Recoleta / Rejeição",
        "setor": "Microbiologia",
        "amostra": "Urina (Urocultura)",
        "motivo": "Contaminação da Amostra",
        "descricao": "Presença de mais de 3 colônias uropatogênicas distintas por falta de assepsia genital prévia",
        "origem_falha": "Orientação ao Paciente / Coleta",
        "gravidade": "Alta",
        "sla_dias": 2
    },
    
    # --- ATRASOS DE LAUDO / TAT (ANALÍTICA E PÓS-ANALÍTICA) ---
    {
        "pilar": "Atraso de Laudo (TAT)",
        "setor": "Microbiologia",
        "amostra": "Urina (Urocultura)",
        "motivo": "Necessidade de Repique (Cultura Mista)",
        "descricao": "Isolamento de colônia pura exigiu 24h adicionais de incubação antes do antibiograma",
        "origem_falha": "Bancada de Microbiologia",
        "gravidade": "Média",
        "sla_dias": 2
    },
    {
        "pilar": "Atraso de Laudo (TAT)",
        "setor": "Micologia",
        "amostra": "Unhas / Pele",
        "motivo": "Crescimento Fúngico Lento",
        "descricao": "Fungos dermatófitos com taxa lenta de duplicação exigindo até 15-30 dias de incubação",
        "origem_falha": "Biológica / Bancada",
        "gravidade": "Baixa",
        "sla_dias": 5
    },
    {
        "pilar": "Atraso de Laudo (TAT)",
        "setor": "Urinálise",
        "amostra": "Urina",
        "motivo": "Gargalo de Microscopia / Sedimento Duvidoso",
        "descricao": "Dúvida em cilindros patológicos ou dismorfismo eritrocitário retendo a liberação",
        "origem_falha": "Bancada de Urinálise",
        "gravidade": "Média",
        "sla_dias": 1
    },
    
    # --- RESULTADOS ERRADOS / RETIFICAÇÕES ---
    {
        "pilar": "Resultado Errado / Retificação",
        "setor": "Urinálise",
        "amostra": "Urina",
        "motivo": "Inversão de Tubos na Centrífuga",
        "descricao": "Troca acidental de sedimento urinário entre dois pacientes vizinhos de caçapa",
        "origem_falha": "Bancada de Urinálise",
        "gravidade": "Crítica",
        "sla_dias": 1
    },
    {
        "pilar": "Resultado Errado / Retificação",
        "setor": "Parasitologia",
        "amostra": "Fezes",
        "motivo": "Confusão Diagnóstica de Cisto",
        "descricao": "Cisto de Entamoeba coli confundido e liberado como Entamoeba histolytica",
        "origem_falha": "Bancada de Parasitologia",
        "gravidade": "Alta",
        "sla_dias": 1
    },
    {
        "pilar": "Resultado Errado / Retificação",
        "setor": "Microbiologia",
        "amostra": "Urina (Urocultura)",
        "motivo": "Inconsistência de Perfil Fenotípico",
        "descricao": "Laudo emitido com bactéria Gram-negativa resistente intrínseca descrita como sensível",
        "origem_falha": "Bancada / Supervisão",
        "gravidade": "Crítica",
        "sla_dias": 1
    }
]

# PASSO 3: Converter a lista em DataFrame do Pandas
df_qualidade = pd.DataFrame(ocorrencias_laboratorio)

# PASSO 4: Exibir informações e os dados
print(f"✓ Sucesso! Total de desvios cadastrados no catálogo: {len(df_qualidade)}")
print("\n--- Resumo por Especialidade ---")
print(df_qualidade["setor"].value_counts())

# Exibir a tabela completa de forma visual
df_qualidade[["pilar", "setor", "amostra", "motivo", "origem_falha", "gravidade"]]


## 📊 4. Insights de Sistemas de Informação para o Power BI

No modelo de dados estrela (**Star Schema**), esta tabela que acabamos de criar no Pandas funcionará como a nossa **Dimensão de Motivos de Falha (`dim_motivo_desvio`)**.

Cada linha tem um identificador único (`id_motivo`) que se conectará à nossa futura **Tabela Fato de Ocorrências (`fato_ocorrencias_qualidade`)**.

Isso permitirá aos gestores no Power BI:
1. Filtrar por especialidade (ex: clicar em *Urinálise* e ver apenas os tubos quebrados e vazamentos);
2. Medir o impacto do erro de cadastro da recepção no retrabalho da bancada;
3. Identificar o ranking de motivos que mais causam recoletas de urina.
